# 02 · Feature Engineering

Builds the cycle-level and cell-level feature tables and inspects how the
engineered features relate to degradation and escalation.

> Synthetic data; no Apple confidential data.


In [ ]:
# Make the project importable when running the notebook from notebooks/
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config
pd.set_option('display.width', 120)

In [ ]:
from src.features.build_features import build as build_features, CYCLE_FEATURE_COLUMNS, CELL_FEATURE_COLUMNS
out = build_features()
cf, cellf = out['cycle_features'], out['cell_features']
print('cycle_features', cf.shape); print('cell_features', cellf.shape)
cf[CYCLE_FEATURE_COLUMNS].describe().T

## Feature correlation with SOH (cycle level)


In [ ]:
num = cf[CYCLE_FEATURE_COLUMNS + ['soh_current']].select_dtypes('number')
corr = num.corr()['soh_current'].drop('soh_current').sort_values()
corr.plot.barh(figsize=(7,5), title='Correlation with SOH'); plt.tight_layout(); plt.show()

## Capacity fade vs resistance growth (cell level)


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sc = ax.scatter(cellf.capacity_fade_rate, cellf.resistance_growth_rate, c=cellf.escalation_required, cmap='coolwarm', s=30, edgecolor='k', lw=.3)
ax.set_xlabel('capacity_fade_rate'); ax.set_ylabel('resistance_growth_rate')
ax.set_title('Escalated cells (red) separate in fade/resistance space'); plt.colorbar(sc, label='escalation'); plt.show()

## Class balance of the escalation target


In [ ]:
cellf['escalation_required'].value_counts().plot.bar(figsize=(5,3), title='escalation_required'); plt.show()

Leakage note: the SOH model deliberately excludes capacity-derived features
and the failure classifier excludes `batch_failure_rate` — see
`src/models/_common.py` and `ai_workflows/model_debugging_workflow.md`.
